In [ ]:
!pip install -q pytabkit

In [ ]:
import numpy as np
import pandas as pd
import gc
import os
import torch
import warnings

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, TargetEncoder
from pytabkit import RealMLP_TD_Classifier
from itertools import combinations

warnings.filterwarnings('ignore')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
if os.path.exists('/kaggle/input'):
    COMP_PATH = '/kaggle/input/competitions/playground-series-s6e3/'
    ORIG_PATH = '/kaggle/input/datasets/cdeotte/s6e3-original-dataset/WA_Fn-UseC_-Telco-Customer-Churn.csv'
    print(f"🔵 Kaggle environment")
else:
    COMP_PATH = 'ChurnMarch/'
    ORIG_PATH = 'ChurnMarch/WA_Fn-UseC_-Telco-Customer-Churn.csv'
    print(f"🟢 Local environment")

train = pd.read_csv(os.path.join(COMP_PATH, 'train.csv'))
test  = pd.read_csv(os.path.join(COMP_PATH, 'test.csv'))

try:
    orig = pd.read_csv(ORIG_PATH)
    print(f"✅ Original dataset loaded: {orig.shape}")
    HAS_ORIG = True
except FileNotFoundError:
    print("⚠️ Original dataset not found, using competition data only")
    HAS_ORIG = False

print(f"Train: {train.shape}, Test: {test.shape}")

In [ ]:
# ---------- Config ----------
TARGET = 'Churn'

cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService',
    'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# ---------- Standardize SeniorCitizen & Target ----------
sc_map = {1: 'Yes', 0: 'No'}
target_map = {'Yes': 1, 'No': 0}

for df in [train, test]:
    df['SeniorCitizen'] = df['SeniorCitizen'].map(sc_map)
    df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
    df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])

train[TARGET] = train[TARGET].map(target_map).astype(int)

# ---------- Merge original data ----------
if HAS_ORIG:
    orig['SeniorCitizen'] = orig['SeniorCitizen'].map(sc_map)
    orig['TotalCharges'] = pd.to_numeric(orig['TotalCharges'], errors='coerce')
    orig['TotalCharges'] = orig['TotalCharges'].fillna(orig['MonthlyCharges'])
    orig[TARGET] = orig[TARGET].map(target_map).astype(int)
    
    # Drop customerID if present, add dummy 'id' for concat
    if 'customerID' in orig.columns:
        orig = orig.drop(columns=['customerID'])
    
    # Align columns — keep only columns present in train
    shared_cols = [c for c in train.columns if c in orig.columns]
    orig_aligned = orig[shared_cols].copy()
    
    # Find cols in train not in orig (like 'id') and fill with dummy values
    for c in train.columns:
        if c not in orig_aligned.columns:
            if c == 'id':
                orig_aligned[c] = range(train['id'].max() + 1, train['id'].max() + 1 + len(orig_aligned))
            else:
                orig_aligned[c] = np.nan
    
    orig_aligned = orig_aligned[train.columns]
    
    # Deduplicate against train using feature columns
    merge_key_cols = [c for c in train.columns if c not in ['id', TARGET]]
    train_keys = train[merge_key_cols].apply(tuple, axis=1)
    orig_keys = orig_aligned[merge_key_cols].apply(tuple, axis=1)
    mask = ~orig_keys.isin(train_keys)
    new_rows = orig_aligned[mask]
    
    train = pd.concat([train, new_rows], ignore_index=True)
    print(f"After merging original: train = {train.shape}")
else:
    print(f"Train (no merge): {train.shape}")

print(f"Target distribution: {train[TARGET].mean():.4f}")

## 3. Feature Engineering — Domain Features

In [ ]:
for df in [train, test]:
    t  = df['tenure'].astype(float)
    mc = df['MonthlyCharges']
    tc = df['TotalCharges']
    
    # --- Charges deviation ---
    df['charges_deviation']      = tc - t * mc
    df['charges_deviation_sign'] = df['charges_deviation'].apply(np.sign).astype('int64')
    df['charges_deviation_abs']  = df['charges_deviation'].abs()
    
    # --- Tenure as category (copy) ---
    df['tenure_2'] = df['tenure'].astype('category')
    
    # --- Tenure features ---
    df['tenure_years']   = (t / 12).apply(np.floor).astype(int)
    df['tenure_segment'] = pd.cut(
        t, bins=[0, 12, 24, 48, 72], labels=[0, 1, 2, 3], include_lowest=True
    ).astype(int)
    df['is_new']   = (t <= 3).astype(int)
    df['is_loyal'] = (t >= 48).astype(int)
    df['tenure_log'] = np.log1p(t)
    
    # --- Charges ratios ---
    df['avg_monthly']       = tc / (t + 1)
    df['charge_ratio']      = mc / (df['avg_monthly'] + 1e-5)
    df['charge_increase']   = mc - df['avg_monthly']
    df['remaining_value']   = mc * (72 - t).clip(lower=0)
    df['expected_total']    = mc * t
    df['total_diff']        = tc - df['expected_total']
    df['is_high_charge']    = (mc > 70).astype(int)
    
    # --- Service counts ---
    internet_svcs = ['OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
                     'TechSupport', 'StreamingTV', 'StreamingMovies']
    df['n_internet_svcs'] = sum((df[c] == 'Yes').astype(int) for c in internet_svcs)
    df['has_phone']    = (df['PhoneService'] == 'Yes').astype(int)
    df['has_internet'] = (df['InternetService'] != 'No').astype(int)
    df['has_fiber']    = (df['InternetService'] == 'Fiber optic').astype(int)
    df['has_security']   = (df['OnlineSecurity'] == 'Yes').astype(int)
    df['has_backup']     = (df['OnlineBackup'] == 'Yes').astype(int)
    df['has_protection'] = (df['DeviceProtection'] == 'Yes').astype(int)
    df['has_support']    = (df['TechSupport'] == 'Yes').astype(int)
    df['n_protect']   = df['has_security'] + df['has_backup'] + df['has_protection'] + df['has_support']
    df['n_stream']    = sum((df[c] == 'Yes').astype(int) for c in ['StreamingTV', 'StreamingMovies'])
    df['total_svcs']  = df['has_phone'] + df['has_internet'] + df['n_internet_svcs']
    df['no_protect']  = (df['n_protect'] == 0).astype(int)
    df['stream_only'] = ((df['n_stream'] > 0) & (df['n_protect'] == 0)).astype(int)
    
    # --- Contract & billing ---
    df['is_mtm']    = (df['Contract'] == 'Month-to-month').astype(int)
    df['is_2yr']    = (df['Contract'] == 'Two year').astype(int)
    df['paperless'] = (df['PaperlessBilling'] == 'Yes').astype(int)
    df['echeck']    = (df['PaymentMethod'] == 'Electronic check').astype(int)
    df['auto_pay']  = df['PaymentMethod'].isin(
        ['Bank transfer (automatic)', 'Credit card (automatic)']
    ).astype(int)
    
    # --- Demographics ---
    df['senior']    = (df['SeniorCitizen'] == 'Yes').astype(int)
    df['partner']   = (df['Partner'] == 'Yes').astype(int)
    df['dependents']  = (df['Dependents'] == 'Yes').astype(int)
    df['family']      = df['partner'] + df['dependents']
    df['senior_alone'] = (df['senior'] & ~df['partner'].astype(bool)).astype(int)
    
    # --- Interaction features (high-signal) ---
    df['fiber_no_protect']  = df['has_fiber'] * df['no_protect']
    df['fiber_mtm']         = df['has_fiber'] * df['is_mtm']
    df['new_fiber']         = df['is_new'] * df['has_fiber']
    df['new_mtm']           = df['is_new'] * df['is_mtm']
    df['echeck_mtm']        = df['echeck'] * df['is_mtm']
    df['hi_charge_mtm']     = df['is_high_charge'] * df['is_mtm']
    df['hi_charge_fiber']   = df['is_high_charge'] * df['has_fiber']
    df['svc_per_charge']    = df['total_svcs'] / (mc + 1e-5)
    df['tenure_x_mc']       = t * mc
    df['tenure_x_mtm']      = t * df['is_mtm']
    df['tenure_x_protect']  = t * df['n_protect']
    df['loyalty']           = t * (1 - df['is_mtm']) * df['auto_pay']
    df['fiber_echeck']      = df['has_fiber'] * df['echeck']
    df['new_echeck']        = df['is_new'] * df['echeck']
    df['fiber_no_support']  = df['has_fiber'] * (1 - df['has_support'])
    
    # --- Risk score ---
    df['risk_v1'] = (
        df['is_mtm'] * 3 + df['has_fiber'] * 2 + df['echeck'] * 2 +
        df['no_protect'] * 2 + df['is_new'] * 3 + df['senior_alone'] * 1 -
        df['is_2yr'] * 3 - df['auto_pay'] * 2 - df['is_loyal'] * 3
    )

print(f"After domain FE: train={train.shape}, test={test.shape}")

## 4. Feature Engineering — N-grams, Rounding, Digits

In [ ]:
# ══════════════════════════════════════════════════════
# N-GRAM FEATURES
# ══════════════════════════════════════════════════════
BIGRAM_COLS = []
TRIGRAM_COLS = []
TOP_CATS_FOR_NGRAM = [
    'Contract', 'InternetService', 'PaymentMethod',
    'OnlineSecurity', 'TechSupport', 'PaperlessBilling'
]
TOP4 = TOP_CATS_FOR_NGRAM[:4]
dataframes = [train, test]

# Bi-grams
for c1, c2 in combinations(TOP_CATS_FOR_NGRAM, 2):
    col_name = f"BG_{c1}_{c2}"
    for df in dataframes:
        df[col_name] = df[c1].astype(str) + "_" + df[c2].astype(str)
    BIGRAM_COLS.append(col_name)

# Tri-grams
for c1, c2, c3 in combinations(TOP4, 3):
    col_name = f"TG_{c1}_{c2}_{c3}"
    for df in dataframes:
        df[col_name] = df[c1].astype(str) + "_" + df[c2].astype(str) + "_" + df[c3].astype(str)
    TRIGRAM_COLS.append(col_name)

NGRAM_COLS = BIGRAM_COLS + TRIGRAM_COLS
for df in dataframes:
    df[NGRAM_COLS] = df[NGRAM_COLS].astype('category')

print(f"Created {len(NGRAM_COLS)} n-gram features")

In [ ]:
train_fe = train.copy()
test_fe  = test.copy()
num_cols2 = num_cols + ['charges_deviation_abs']

# --- Rounding ---
round_config = {
    'MonthlyCharges'        : [-1],
    'TotalCharges'          : [-2, -3],
    'charges_deviation_abs' : [-2, -3]
}
ROUND = []
for col, r_values in round_config.items():
    for r in r_values:
        feat = f"{col}_r{r}"
        for df in [train_fe, test_fe]:
            df[feat] = df[col].round(r)
        ROUND.append(feat)

# --- Digit extraction ---
digit_config = {
    'tenure'                : [-1, 0],
    'MonthlyCharges'        : [-2, -1, 0, 1, 2],
    'TotalCharges'          : [-3, -2, -1, 0, 1, 2],
    'charges_deviation_abs' : [-3, -2, -1, 0, 1, 2]
}
DIGITS = []
for col, k_values in digit_config.items():
    for k in k_values:
        feat = f"{col}_d{k}"
        for df in [train_fe, test_fe]:
            df[feat] = ((df[col] * 10**k) % 10).astype(int)
        DIGITS.append(feat)

# --- Decimal features ---
DECIMALS = []
for col in ['MonthlyCharges', 'TotalCharges', 'charges_deviation_abs']:
    feat = f"{col}_decimal"
    for df in [train_fe, test_fe]:
        df[feat] = (df[col] % 1).round(2)
    DECIMALS.append(feat)

# --- Bin features ---
BINS = []
for col in num_cols2:
    bin_col = f"{col}_bin"
    train_fe[bin_col], bin_edges = pd.qcut(
        train_fe[col], q=10, labels=False, duplicates='drop', retbins=True
    )
    test_fe[col] = test_fe[col].clip(bin_edges[0], bin_edges[-1])
    test_fe[bin_col] = pd.cut(
        test_fe[col], bins=bin_edges, labels=False, include_lowest=True
    ).fillna(0).astype(int)
    BINS.append(bin_col)

# --- Numerical interactions ---
for col1, col2 in combinations(num_cols2, 2):
    for df in [train_fe, test_fe]:
        df[f"{col1}_mul_{col2}"] = df[col1] * df[col2]
        df[f"{col1}_div_{col2}"] = df[col1] / (df[col2] + 1e-6)
        df[f"{col2}_div_{col1}"] = df[col2] / (df[col1] + 1e-6)

# --- Squared features ---
for col in num_cols2:
    for df in [train_fe, test_fe]:
        df[f"{col}_sq"] = df[col] ** 2

print(f"After numeric FE: train_fe={train_fe.shape}, test_fe={test_fe.shape}")

## 5. Preprocessing

In [ ]:
for df in [train_fe, test_fe]:
    for col in cat_cols:
        df[col] = df[col].astype('category')

all_num_cols = (
    num_cols + 
    ['charges_deviation', 'charges_deviation_abs', 'avg_monthly', 'charge_ratio',
     'charge_increase', 'remaining_value', 'expected_total', 'total_diff',
     'tenure_log', 'svc_per_charge', 'tenure_x_mc', 'risk_v1', 'loyalty'] +
    ROUND + DECIMALS
)

all_num_cols = [c for c in all_num_cols if c in train_fe.columns]

scaler = StandardScaler()
train_fe[all_num_cols] = scaler.fit_transform(train_fe[all_num_cols])
test_fe[all_num_cols]  = scaler.transform(test_fe[all_num_cols])

# Prepare X, y
X = train_fe.drop([TARGET, 'id'], axis=1)
y = train_fe[TARGET]
X_test = test_fe.drop('id', axis=1)

print(f"X: {X.shape}, y: {y.shape}, X_test: {X_test.shape}")
print(f"Categorical features: {sum(X[c].dtype.name == 'category' for c in X.columns)}")
print(f"Numerical features: {sum(X[c].dtype.name != 'category' for c in X.columns)}")

## 6. RealMLP Configuration

In [ ]:
SEEDS = [42, 123]
N_FOLDS = 5

common_fixed_params = {
    'act':                    'mish',
    'block_str':              'w-b-a-d',
    'hidden_sizes':           'rectangular',
    'embedding_size':         8,
    'max_one_hot_cat_size':   9,
    'num_emb_type':           'pbld',
    'add_front_scale':        True,
    'use_parametric_act':     True,
    
    # Weight initialization
    'weight_param':           'ntk',
    'weight_init_mode':       'std',
    'bias_init_mode':         'he+5',
    
    # Learning rate scheduling
    'ls_eps_sched':           'coslog4',
    'p_drop_sched':           'flat_cos',
    'wd_sched':               'flat_cos',
    'bias_wd_factor':         0.0,
    'bias_lr_factor':         0.1,
    'act_lr_factor':          0.1,
    
    # Validation & training
    'val_metric_name':        '1-auc_ovr',
    'batch_size':             1024,
    'n_epochs':               5,
    'verbosity':              2,
    
    # Early stopping
    'early_stopping_additive_patience':        10,
    'early_stopping_multiplicative_patience':  3,
    
    # Internal ensemble (8 models per fit)
    'n_ens':                  8,
    'ens_av_before_softmax':  False,
    
    # Device
    'device':                 DEVICE,
}

param_grid = {
    **common_fixed_params,
    'n_hidden_layers':        3,
    'hidden_width':           512,
    'embedding_size':         16,
    'lr':                     0.024
    'wd':                     0.001,
    'p_drop':                 0.3,
    'sq_mom':                 0.1,
    'scale_lr_factor':        1.04,
    'plr_sigma':              5.65,
    'plr_lr_factor':          0.41,
    'max_one_hot_cat_size':   13,
}

print(f"Config: {len(SEEDS)} seeds × {N_FOLDS} folds = {len(SEEDS) * N_FOLDS} models")
print(f"   Each with n_ens={param_grid['n_ens']} → {len(SEEDS) * N_FOLDS * param_grid['n_ens']} effective models")
print(f"   Architecture: {param_grid['n_hidden_layers']} layers × {param_grid['hidden_width']} wide")
print(f"   Device: {DEVICE}")

## 7. Multi-Seed Training Loop

In [ ]:
# Columns to target-encode and then drop raw n-grams
TE_INPUT_COLS  = X.columns.to_list()
TE_OUTPUT_COLS = [f"{col}_te" for col in TE_INPUT_COLS]
COLS_TO_DROP_AFTER_TE = BIGRAM_COLS + TRIGRAM_COLS

oof_preds  = np.zeros(len(X))
test_preds = np.zeros(len(X_test))
all_fold_scores = []

for seed_idx, seed in enumerate(SEEDS):
    print(f"\n{'='*60}")
    print(f"🌱 Seed {seed} ({seed_idx + 1}/{len(SEEDS)})")
    print(f"{'='*60}")
    
    # Update random state in params
    param_grid['random_state'] = seed
    
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    seed_fold_scores = []
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n--- Seed {seed} | Fold {fold + 1}/{N_FOLDS} ---")
        
        X_tr, X_val = X.iloc[train_idx].copy(), X.iloc[val_idx].copy()
        y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
        X_te = X_test.copy()
        
        # --- Target Encoding ---
        te = TargetEncoder(smooth='auto', cv=5, shuffle=True, random_state=seed)
        X_tr[TE_OUTPUT_COLS]  = te.fit_transform(X_tr[TE_INPUT_COLS], y_tr)
        X_val[TE_OUTPUT_COLS] = te.transform(X_val[TE_INPUT_COLS])
        X_te[TE_OUTPUT_COLS]  = te.transform(X_te[TE_INPUT_COLS])
        
        # --- Drop raw N-grams (too many categories for embedding) ---
        X_tr.drop(columns=COLS_TO_DROP_AFTER_TE, inplace=True)
        X_val.drop(columns=COLS_TO_DROP_AFTER_TE, inplace=True)
        X_te.drop(columns=COLS_TO_DROP_AFTER_TE, inplace=True)
        
        # --- Train ---
        model = RealMLP_TD_Classifier(**param_grid)
        model.fit(X_tr, y_tr.values, X_val, y_val.values)
        
        # --- Predict ---
        val_probs      = model.predict_proba(X_val)[:, 1]
        fold_test_probs = model.predict_proba(X_te)[:, 1]
        
        oof_preds[val_idx] += val_probs / len(SEEDS)
        test_preds         += fold_test_probs / (N_FOLDS * len(SEEDS))
        
        score = roc_auc_score(y_val, val_probs)
        seed_fold_scores.append(score)
        all_fold_scores.append(score)
        print(f"   Fold {fold + 1} ROC-AUC: {score:.6f}")
        
        # --- Cleanup ---
        del model, X_tr, X_val, X_te, y_tr, y_val
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    seed_mean = np.mean(seed_fold_scores)
    print(f"\n📊 Seed {seed} Mean AUC: {seed_mean:.6f} ± {np.std(seed_fold_scores):.5f}")


final_auc = roc_auc_score(y, oof_preds)
print(f"\n{'='*60}")
print(f"FINAL OOF ROC-AUC: {final_auc:.6f}")
print(f"   Mean Fold AUC:     {np.mean(all_fold_scores):.6f} ± {np.std(all_fold_scores):.5f}")
print(f"   Models trained:    {len(SEEDS) * N_FOLDS}")
print(f"{'='*60}")

## 8. Submission

In [ ]:
submission = pd.DataFrame({
    'id': test['id'],
    'Churn': test_preds
})

out_path = '/kaggle/working/submission.csv' if os.path.exists('/kaggle/working') else 'submission.csv'
submission.to_csv(out_path, index=False)

print(f"Saved submission to {out_path}")
print(f"   Shape: {submission.shape}")
print(f"\n{submission.head(10)}")
print(f"\nPrediction stats:\n{submission['Churn'].describe()}")